In [1]:
from pathlib import Path

from imagematerials.vehicles import (
    preprocess,
)
from imagematerials.util import import_from_netcdf, export_to_netcdf
from imagematerials.model import GenericMainModel, GenericMaterials, GenericStocks
from imagematerials.factory import ModelFactory
from imagematerials.maintenance import Maintenance
import prism
import warnings


In [2]:
base_dir = "../data/raw"
prep_fp = Path("prep_vema.nc")

In [3]:
if not prep_fp.is_file():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        orig_prep_data = preprocess(base_dir)
    export_to_netcdf(orig_prep_data, prep_fp)
prep_data = import_from_netcdf(prep_fp)
prep_data["weights"] = prep_data.pop("vehicle_weights")

In [4]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [5]:
# Define the coordinates of all dimensions.
Region = list(prep_data["stocks"].coords["Region"].values)
Time = [t for t in complete_timeline]
Cohort = Time
Type = list(prep_data["stocks"].coords["Type"].values)
material = list(prep_data["material_fractions"].coords["material"].values)

# Create
main_model_normal = GenericMainModel(
    complete_timeline, Region=Region, Time=Time, Cohort=Cohort, Type=Type, prep_data=prep_data,
    compute_materials=True, compute_battery_materials=False, compute_maintenance_materials=False, 
    material=material)

In [6]:
main_model_normal.simulate(simulation_timeline)

IndexError: dimension coordinate 'Type' conflicts between indexed and indexing objects:
<xarray.DataArray 'Type' (Type: 7)> Size: 980B
array(['Bikes', 'Freight Planes', 'Freight Trains', 'High Speed Trains',
       'Inland Ships', 'Passenger Planes', 'Trains'], dtype='<U35')
Coordinates:
  * Type     (Type) <U35 980B 'Bikes' 'Freight Planes' ... 'Trains'
    Time     int64 8B 1961
vs.
<xarray.IndexVariable 'Type' (Type: 17)> Size: 2kB
array(['Bikes', 'Cars', 'Freight Planes', 'Freight Trains',
       'Heavy Freight Trucks', 'High Speed Trains', 'Inland Ships',
       'Large Ships', 'Light Commercial Vehicles', 'Medium Freight Trucks',
       'Medium Ships', 'Midi Buses', 'Passenger Planes', 'Regular Buses',
       'Small Ships', 'Trains', 'Very Large Ships'], dtype='<U25')

In [ ]:
main_model_factory = ModelFactory(
    prep_data, complete_timeline
    ).add(GenericStocks
    ).add(GenericMaterials
    ).finish()

ValueError: Mismatched dimensions in input data for battery_weights for coordinates Type

In [ ]:
main_model_factory.simulate(simulation_timeline)

In [ ]:
prep_data["battery_weights"]

<xarray.DataArray (Cohort: 254, Type: 36)> Size: 73kB
array([[240. ,   0. ,  20.8, ...,   0. , 194.1, 118. ],
       [240. ,   0. ,  20.8, ...,   0. , 194.1, 118. ],
       [240. ,   0. ,  20.8, ...,   0. , 194.1, 118. ],
       ...,
       [240. ,   0. ,  20.8, ...,   0. , 194.1, 118. ],
       [240. ,   0. ,  20.8, ...,   0. , 194.1, 118. ],
       [240. ,   0. ,  20.8, ...,   0. , 194.1, 118. ]])
Coordinates:
  * Cohort   (Cohort) int64 2kB 1807 1808 1809 1810 1811 ... 2057 2058 2059 2060
  * Type     (Type) <U35 5kB 'Cars - BEV' ... 'Regular Buses - Trolley'